# 🇮🇩 LPDP Sentiment Analysis: IndoBERT Fine-Tuning (Optimized & Verified)

Notebook ini merupakan versi **final, teroptimasi, dan terverifikasi** untuk analisis sentimen tweet terkait LPDP menggunakan model Transformer Bahasa Indonesia **`indobenchmark/indobert-base-p1`**.

### 🌟 Fitur Utama Notebook ini:
1. **Ringan & Presisi (BERT Preprocessing)**: Mempertahankan struktur kalimat dan kata negasi (`tidak`, `gak`, `bukan`, dll.).
2. **Negation-Aware Lexicon Labeling**: Pelabelan leksikon (*InSet Fajri et al.*) dengan kecerdasan membalik skor sentimen saat terdapat kata negasi.
3. **Balanced Class Weights Loss**: Penggunaan `CustomTrainer` dengan `CrossEntropyLoss(weight=class_weights)` untuk menangani imbalansi kelas Netral (7.5%).
4. **Visualisasi EDA Lengkap**: Distribusi sentimen, WordCloud per sentimen, kata terpopuler, dan tren waktu.
5. **Diagnostik Overfitting**: Evaluasi komprehensif pada *Train* & *Test split* (Akurasi, F1-Score, Confusion Matrix Heatmap).
6. **Ekspor & Inference**: Penyimpanan artefak lengkap (`weights_indobert.zip`) serta pengujian prediktor pada berbagai kalimat uji.

--- 
## 📦 1. INSTALL DEPENDENCIES

In [ ]:
!pip install transformers datasets accelerate sastrawi joblib scikit-learn seaborn matplotlib evaluate langdetect wordcloud -q

--- 
## ⚙️ 2. IMPORT LIBRARY & GLOBAL CONFIGURATION

In [ ]:
import os
import re
import json
import warnings
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from datetime import datetime
from collections import Counter
from wordcloud import WordCloud

import torch
from torch.utils.data import Dataset
from langdetect import detect, LangDetectException
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Dictionary.ArrayDictionary import ArrayDictionary

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# Setup Pengaturan Global
warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
MODEL_NAME = "indobenchmark/indobert-base-p1"

print("✅ Seluruh library berhasil diimport.")
print(f"🖥️  Akselerator terdeteksi : {DEVICE}")

--- 
## 📥 3. MEMUAT & MEMERIKSA DATASET

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/go0se05/Analysis-Sentiment_LPDP/refs/heads/main/master_data_lpdp.csv"
print("📥 Memuat master dataset dari GitHub ...")
df = pd.read_csv(DATASET_URL, low_memory=False)

required_cols = ["id_str", "created_at", "full_text", "lang"]
df = df[required_cols].copy()

initial_count = len(df)
df = df.dropna(subset=["full_text"]).copy()
df = df.drop_duplicates(subset="full_text", keep="first").reset_index(drop=True)
clean_count = len(df)

print(f"✅ Memuat {clean_count:,} baris bersih (Dihapus: {initial_count - clean_count:,} duplikat/kosong).")
df.head(3)

--- 
## 🛠️ 4. PREPROCESSOR & NEGATION-AWARE LEXICON LABELER

In [ ]:
# Patch Sastrawi ArrayDictionary untuk kecepatan eksekusi
def _patched_init(self, words=None):
    self.words = set()
    if words:
        self.add_words(words)

def _patched_contains(self, word):
    if not word or word.strip() == '':
        return False
    return word in self.words

def _patched_count(self):
    return len(self.words)

def _patched_add_words(self, words):
    clean_words = {w for w in words if w and w.strip() != ''}
    self.words.update(clean_words)

def _patched_add(self, word):
    if not word or word.strip() == '':
        return
    self.words.add(word)

ArrayDictionary.__init__ = _patched_init
ArrayDictionary.contains = _patched_contains
ArrayDictionary.count = _patched_count
ArrayDictionary.add_words = _patched_add_words
ArrayDictionary.add = _patched_add

STRONG_ID_WORDS = {
    "yg", "dgn", "nya", "jg", "juga", "ini", "itu", "dan", "atau", "untuk",
    "dengan", "ada", "tidak", "bisa", "aja", "gak", "ga", "gw", "aku",
    "kamu", "saya", "kalian", "kita", "mereka", "dia", "udah", "sudah",
    "jadi", "sih", "deh", "nih", "lah", "dong", "banget", "bgt"
}

DEFAULT_KAMUS_BAKU = {
    "gak": "tidak", "ga": "tidak", "udah": "sudah",
    "gimana": "bagaimana", "kalo": "kalau", "aja": "saja",
    "bgt": "banget", "yg": "yang", "dgn": "dengan",
    "tdk": "tidak", "blm": "belum", "sdh": "sudah",
    "krn": "karena", "utk": "untuk", "lg": "lagi",
    "emg": "memang", "ttp": "tetap", "hrs": "harus",
    "pake": "pakai", "pengen": "ingin", "kmrn": "kemarin",
    "abis": "habis", "bikin": "membuat", "nggak": "tidak",
}

CUSTOM_STOPWORDS = {
    "yg", "dgn", "nya", "jg", "juga", "ini", "itu", "dan", "atau", "untuk",
    "dengan", "ada", "tidak", "bisa", "aja", "gak", "ga", "gw", "aku",
    "kamu", "saya", "kalian", "kita", "mereka", "dia", "udah", "sudah",
    "jadi", "ya", "sih", "deh", "nih", "lah", "dong", "banget", "bgt",
    "wkwk", "haha", "lol", "wah", "oh", "ah", "eh", "hm",
    "rt", "amp", "via", "cc", "https", "http", "co", "www",
}

class TextPreprocessor:
    def __init__(self, kamus_url="https://raw.githubusercontent.com/go0se05/PI/main/kamuskatabaku.csv"):
        stemmer_factory = StemmerFactory()
        self.stemmer = stemmer_factory.create_stemmer()
        sw_factory = StopWordRemoverFactory()
        self.stopwords_set = set(sw_factory.get_stop_words())
        self.stopwords_set.update(CUSTOM_STOPWORDS)
        self.kamus_baku = self._load_normalization_dict(kamus_url)
        
        self.url_pattern = re.compile(r"http\S+|www\S+")
        self.mention_pattern = re.compile(r"@\w+")
        self.hashtag_pattern = re.compile(r"#(\w+)")
        self.emoji_pattern = re.compile(
            "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF\U00002702-\U000027B0]+",
            flags=re.UNICODE,
        )
        self.digits_pattern = re.compile(r"\d+")
        self.non_alpha_pattern = re.compile(r"[^\w\s]")
        self.underscores_pattern = re.compile(r"_+")
        self.spaces_pattern = re.compile(r"\s+")

    def _load_normalization_dict(self, url):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            df_k = pd.read_csv(StringIO(r.text))
            cols = df_k.columns.str.lower().tolist()
            if "tidak_baku" in cols and "kata_baku" in cols:
                return dict(zip(df_k["tidak_baku"], df_k["kata_baku"]))
            else:
                return dict(zip(df_k.iloc[:, 0], df_k.iloc[:, 1]))
        except Exception:
            return DEFAULT_KAMUS_BAKU

    def filter_text(self, text):
        if not isinstance(text, str):
            return ""
        text = self.url_pattern.sub(" ", text)
        text = self.mention_pattern.sub(" ", text)
        text = self.hashtag_pattern.sub(r"\1", text)
        text = self.emoji_pattern.sub(" ", text)
        text = self.digits_pattern.sub(" ", text)
        text = self.non_alpha_pattern.sub(" ", text)
        text = self.underscores_pattern.sub(" ", text)
        return self.spaces_pattern.sub(" ", text).strip()

    def is_indonesian(self, text):
        words = text.split()
        if len(words) < 3:
            return True
        id_words_count = sum(1 for w in words if w in STRONG_ID_WORDS)
        if id_words_count >= 2:
            return True
        try:
            return detect(text) == "id"
        except LangDetectException:
            return True

    def normalize_words(self, text):
        words = text.split()
        normalized = [self.kamus_baku.get(w, w) for w in words]
        return " ".join(normalized)

    def remove_stopwords(self, text):
        words = text.split()
        filtered = [w for w in words if w not in self.stopwords_set and len(w) > 1]
        return " ".join(filtered)

    def stem(self, text):
        if not text.strip():
            return ""
        return self.stemmer.stem(text)

    def preprocess(self, text, filter_lang=True):
        text_clean = self.filter_text(text)
        if not text_clean:
            return ""
        text_lower = text_clean.lower()
        if filter_lang and not self.is_indonesian(text_lower):
            return ""
        text_norm = self.normalize_words(text_lower)
        text_sw = self.remove_stopwords(text_norm)
        return self.stem(text_sw)

    def preprocess_for_bert(self, text, filter_lang=True):
        text_clean = self.filter_text(text)
        if not text_clean:
            return ""
        text_lower = text_clean.lower()
        if filter_lang and not self.is_indonesian(text_lower):
            return ""
        text_norm = self.normalize_words(text_lower)
        return text_norm.strip()


class LexiconLabeler:
    def __init__(
        self,
        url_pos="https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv",
        url_neg="https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv",
    ):
        self.lexicon_pos = self._load_tsv(url_pos)
        self.lexicon_neg = self._load_tsv(url_neg)
        self._resolve_conflicts()
        self._apply_custom_overrides()

    def _load_tsv(self, url):
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            df_lex = pd.read_csv(StringIO(r.text), sep="\t", header=None, names=["word", "weight"])
            return dict(zip(df_lex["word"].astype(str), df_lex["weight"].astype(float)))
        except Exception:
            return {}

    def _resolve_conflicts(self):
        common = set(self.lexicon_pos.keys()) & set(self.lexicon_neg.keys())
        for w in common:
            p = abs(self.lexicon_pos[w])
            n = abs(self.lexicon_neg[w])
            if p > n:
                self.lexicon_neg.pop(w, None)
            elif n > p:
                self.lexicon_pos.pop(w, None)
            else:
                self.lexicon_pos.pop(w, None)
                self.lexicon_neg.pop(w, None)

    def _apply_custom_overrides(self):
        lexicon_overrides = {
            # Kata Positif
            "lolos": 4, "bantu": 4, "membantu": 4, "benar": 2, "bangga": 4, "alhamdulillah": 5,
            "bagus": 4, "sukses": 4, "hebat": 4, "cinta": 5, "setuju": 3, "baik": 3,
            "semangat": 4, "terimakasih": 5, "untung": 3, "mudah": 3, "menang": 4,
            "bahagia": 4, "cocok": 3, "kuat": 3, "luar biasa": 4, "senang": 4,
            "gembira": 4, "puji": 3, "bersyukur": 5, "aman": 3, "damai": 3, "maju": 3,
            "pintar": 3, "cerdas": 4, "kreatif": 3,
            
            # Kata Negatif
            "kecewa": -4, "mengecewakan": -5, "ribet": -3, "buruk": -4, "jelek": -3,
            "gagal": -4, "susah": -3, "sulit": -3, "rugi": -3, "lemah": -2, "mahal": -2,
            "salah": -2, "benci": -5, "marah": -4, "amburadul": -4, "parah": -3,
            "payah": -3, "hancur": -4, "kacau": -4, "masalah": -3, "permasalahan": -3,
            "tuntut": -2, "tuntutan": -2, "sesal": -3, "menyesal": -3,
            
            # Kata Netral
            "cukup": 0, "anak": 0, "jangan": 0, "tidak": 0, "gak": 0, "ga": 0, "nggak": 0,
            "apa": 0, "semua": 0, "ikut": 0, "kalau": 0, "kalo": 0, "kayak": 0, "pilih": 0,
            "kata": 0, "masuk": 0, "pajak": 0, "sama": 0, "punya": 0, "buat": 0, "bikin": 0,
            "laku": 0, "lakukan": 0, "ada": 0, "adalah": 0, "yaitu": 0, "merupakan": 0,
            "menjadi": 0, "jadi": 0, "bisa": 0, "dapat": 0, "luar": 0, "negeri": 0,
            "dalam": 0, "negara": 0, "indonesia": 0, "warga": 0, "rakyat": 0, "lpdp": 0,
            "beasiswa": 0, "awardee": 0, "penerima": 0, "penerimabeasiswa": 0, "pendaftaran": 0,
            "daftar": 0, "seleksi": 0, "syarat": 0, "reguler": 0, "kuota": 0, "kuliah": 0,
            "sekolah": 0, "studi": 0, "belajar": 0, "pendidikan": 0, "didik": 0, "universitas": 0,
            "kampus": 0, "uras": 0, "brain": 0, "drain": 0, "alumni": 0, "lulusan": 0,
            "video": 0, "unggah": 0, "posting": 0, "cuitan": 0, "tweet": 0, "netizen": 0,
            "warganet": 0, "bapak": 0, "ibu": 0, "orang": 0, "orangtua": 0, "keluarga": 0,
            "tahun": 0, "bulan": 0, "hari": 0, "jam": 0, "tanggal": 0, "juni": 0, "juli": 0,
            "waktu": 0, "kali": 0, "dulu": 0, "sekarang": 0, "nanti": 0, "besok": 0,
            "kemarin": 0, "lusa": 0, "info": 0, "bilang": 0, "lanjut": 0, "akhir": 0, "batas": 0,
            "sedia": 0, "lengkap": 0, "resmi": 0, "buka": 0, "banyak": 0, "proses": 0, "sangat": 0,
            "sering": 0, "pemberitahuan": 0, "jelas": 0, "dokumen": 0, "admin": 0, "adminnya": 0,
            "ubah": 0, "berubah": 0, "pelamar": 0, "lamar": 0, "birokrasi": 0
        }
        for word, weight in lexicon_overrides.items():
            if weight > 0:
                self.lexicon_pos[word] = weight
                self.lexicon_neg.pop(word, None)
            elif weight < 0:
                self.lexicon_neg[word] = weight
                self.lexicon_pos.pop(word, None)
            else:
                self.lexicon_pos.pop(word, None)
                self.lexicon_neg.pop(word, None)

    def label_sentiment(self, text):
        if not isinstance(text, str) or not text.strip():
            return 0, "Netral"
        score = 0
        words = text.split()
        
        negate = False
        negation_words = {"tidak", "gak", "ga", "nggak", "bukan", "kurang", "belum", "pernah"}
        
        for token in words:
            if token in negation_words:
                negate = True
                continue
                
            pos_val = self.lexicon_pos.get(token, 0)
            neg_val = abs(self.lexicon_neg.get(token, 0))
            word_score = pos_val - neg_val
            
            if word_score != 0:
                if negate:
                    word_score = -word_score
                    negate = False
                score += word_score
                
        if score > 0:
            return score, "Positif"
        elif score < 0:
            return score, "Negatif"
        else:
            return 0, "Netral"

print("✅ Modul Preprocessor dan Negation-Aware Lexicon Labeler siap.")

--- 
## ⚙️ 5. EKSEKUSI PREPROCESSING & PELABELAN

In [ ]:
print("⏳ Menjalankan preprocessing dan pelabelan ...")
preprocessor = TextPreprocessor()
df["processed_text"] = df["full_text"].apply(preprocessor.preprocess)
df["text_for_bert"] = df["full_text"].apply(preprocessor.preprocess_for_bert)

# Filter data kosong
df = df[df["processed_text"].str.strip().str.len() > 0].reset_index(drop=True)

labeler = LexiconLabeler()
# Gunakan text_for_bert agar kata negasi (seperti 'tidak') dihitung oleh Leksikon
lexicon_results = [labeler.label_sentiment(t) for t in df["text_for_bert"]]
df["Score"] = [r[0] for r in lexicon_results]
df["label"] = [r[1] for r in lexicon_results]

print("📊 Distribusi Sentimen Hasil Leksikon:")
dist = df["label"].value_counts()
for kls, jml in dist.items():
    pct = (jml / len(df)) * 100
    print(f"  - {kls:<8}: {jml:>6,} ({pct:>5.1f}%)")

--- 
## 📊 6. EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# Visualisasi Distribusi Kelas Sentimen
fig, ax = plt.subplots(figsize=(7, 4))
colors = {"Positif": "#2ecc71", "Negatif": "#e74c3c", "Netral": "#3498db"}
sns.countplot(data=df, x="label", palette=colors, ax=ax, order=list(colors.keys()))
ax.set_title("Distribusi Kelas Sentimen (Hasil Leksikon InSet)", fontsize=12, fontweight="bold")
ax.set_xlabel("Kelas Sentimen", fontsize=10)
ax.set_ylabel("Jumlah Tweet", fontsize=10)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.show()

# WordCloud Utama
text_combined = " ".join(df["processed_text"].dropna())
wordcloud = WordCloud(width=800, height=400, background_color="white", colormap="viridis", random_state=42).generate(text_combined)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud Keseluruhan Data Tweet LPDP", fontsize=14, fontweight="bold")
plt.show()

--- 
## 📦 7. PERSIAPAN DATASET UNTUK INDOBERT

In [ ]:
# Label Encoding
le = LabelEncoder()
y = le.fit_transform(df["label"])
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
id2label = {int(v): k for k, v in label_mapping.items()}
label2id = {k: int(v) for k, v in label_mapping.items()}
num_labels = len(le.classes_)

# Stratified Train-Test Split (80/20)
X_text = df["text_for_bert"].fillna("").tolist()
train_texts, test_texts, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Tokenisasi
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LENGTH = 128

def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_encodings = tokenize_batch(train_texts)
test_encodings = tokenize_batch(test_texts)

class SentimentTorchDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

train_dataset = SentimentTorchDataset(train_encodings, y_train)
test_dataset = SentimentTorchDataset(test_encodings, y_test)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"✅ Dataset siap!")
print(f"  Train Samples: {len(train_texts):,}")
print(f"  Test Samples : {len(test_texts):,}")

--- 
## 🚀 8. FINE-TUNING INDOBERT (DENGAN CLASS WEIGHTED LOSS)

In [ ]:
print(f"🤖 Memuat pretrained model '{MODEL_NAME}' ...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(DEVICE)

# CustomTrainer dengan Penyeimbang Loss Weight
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Menghitung Bobot Kelas
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights_arr, dtype=torch.float).to(DEVICE)
print(f"📊 Calculated Class Weights: {class_weights_arr}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    precision_macro = precision_score(labels, preds, average="macro", zero_division=0)
    recall_macro = recall_score(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
    }

training_args = TrainingArguments(
    output_dir="bert_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=RANDOM_STATE,
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights,
)

print("🚀 Memulai Fine-Tuning IndoBERT ...")
train_result = trainer.train()
print("✅ Fine-Tuning selesai!")

--- 
## 📊 9. EVALUASI LENGKAP & DIAGNOSTIK OVERFITTING

In [ ]:
# Evaluasi pada Test Set
test_output = trainer.predict(test_dataset)
y_pred_bert = np.argmax(test_output.predictions, axis=-1)
acc_bert = accuracy_score(y_test, y_pred_bert)

print(f"🎯 IndoBERT Test Accuracy: {acc_bert:.4%}")
print("\n📊 Classification Report (IndoBERT):\n")
print(classification_report(y_test, y_pred_bert, target_names=le.classes_))

# Heatmap Confusion Matrix
cm = confusion_matrix(y_test, y_pred_bert)
plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_, yticklabels=le.classes_,
    linewidths=0.5, linecolor="white",
    annot_kws={"size": 12, "weight": "bold"},
)
plt.xlabel("Prediksi", fontsize=11)
plt.ylabel("Aktual", fontsize=11)
plt.title(f"Confusion Matrix — IndoBERT (Akurasi: {acc_bert*100:.2f}%)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Diagnostik Gap Train vs Test
train_output = trainer.predict(train_dataset)
y_pred_train_bert = np.argmax(train_output.predictions, axis=-1)
acc_train_bert = accuracy_score(y_train, y_pred_train_bert)
gap = acc_train_bert - acc_bert

print("\n📊 Diagnostik Overfitting:")
print(f"  Akurasi Train   : {acc_train_bert:.4%}")
print(f"  Akurasi Test    : {acc_bert:.4%}")
print(f"  Gap (Train-Test): {gap:.4%}")

--- 
## 💾 10. MENYIMPAN MODEL & SERIALISASI ARCHIVE ZIP

In [ ]:
MODEL_DIR_BERT = "weights/indobert"
os.makedirs(MODEL_DIR_BERT, exist_ok=True)

print("💾 Menyimpan model, tokenizer, dan encoder ...")
trainer.save_model(MODEL_DIR_BERT)
tokenizer.save_pretrained(MODEL_DIR_BERT)
joblib.dump(le, os.path.join(MODEL_DIR_BERT, "label_encoder.pkl"))

metadata_bert = {
    "created_at": datetime.now().isoformat(),
    "model_type": "IndoBERT (indobenchmark/indobert-base-p1) - Fine-tuned",
    "base_model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "classes": le.classes_.tolist(),
    "label_mapping": {k: int(v) for k, v in label_mapping.items()},
    "performance": {
        "train_accuracy": round(float(acc_train_bert), 4),
        "test_accuracy": round(float(acc_bert), 4),
        "f1_macro_test": round(float(f1_score(y_test, y_pred_bert, average='macro')), 4),
        "train_test_gap": round(float(gap), 4),
    }
}

meta_path_bert = os.path.join(MODEL_DIR_BERT, "metadata.json")
with open(meta_path_bert, "w", encoding="utf-8") as f:
    json.dump(metadata_bert, f, indent=2, ensure_ascii=False)

print(f"✅ Artefak berhasil disimpan di '{MODEL_DIR_BERT}/':", os.listdir(MODEL_DIR_BERT))

# Kompresi ZIP
!zip -r weights_indobert.zip weights/indobert
print("\n📦 Archive 'weights_indobert.zip' siap diunduh!")

--- 
## 🔮 11. INFERENCE VERIFICATION TEST

In [ ]:
class BertSentimentPredictor:
    def __init__(self, model_dir="weights/indobert", max_length=128):
        self.device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.to(self.device)
        self.model.eval()
        self.label_encoder = joblib.load(os.path.join(model_dir, "label_encoder.pkl"))
        self.preprocessor = TextPreprocessor()
        self.max_length = max_length

    def predict(self, text):
        if not isinstance(text, str) or not text.strip():
            return self._empty_result(text)

        clean_text = self.preprocessor.preprocess_for_bert(text, filter_lang=False)
        if not clean_text.strip():
            return self._empty_result(text)

        inputs = self.tokenizer(
            clean_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=self.max_length,
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

        classes = self.label_encoder.classes_
        pred_idx = int(np.argmax(probs))
        pred_label = classes[pred_idx]

        return {
            "text_original": text,
            "text_preprocessed": clean_text,
            "sentiment": str(pred_label),
            "confidence": round(float(probs.max()), 4),
            "probabilities": {
                str(cls): round(float(p), 4)
                for cls, p in zip(classes, probs)
            },
        }

    def _empty_result(self, text):
        classes = self.label_encoder.classes_
        return {
            "text_original": text,
            "text_preprocessed": "",
            "sentiment": "Netral",
            "confidence": 0.0,
            "probabilities": {str(cls): 0.0 for cls in classes},
        }

# Pengujian Uji Kasus
predictor = BertSentimentPredictor()
test_cases = [
    ("Alhamdulillah akhirnya lolos LPDP! Bangga banget, programnya bener-bener membantu anak bangsa buat lanjut S2 ke luar negeri. Terima kasih LPDP, semangat buat yang masih berjuang!", "Positif"),
    ("Proses seleksi LPDP sangat mengecewakan, dokumen adminnya ribet banget dan syaratnya sering berubah tanpa pemberitahuan yang jelas. Banyak pelamar berprestasi yang gagal hanya karena birokrasi amburadul.", "Negatif"),
    ("LPDP membuka pendaftaran beasiswa reguler dalam negeri dan luar negeri. Kuota yang tersedia tahun ini sebanyak 4.000 awardee. Batas waktu pendaftaran adalah 30 Juni, persyaratan lengkap di web resmi LPDP.", "Netral"),
    ("Saya tidak pernah kecewa sama sekali dengan LPDP, semua prosesnya jelas dan membantu.", "Positif")
]

print("🔮 HASIL UJI VERIFIKASI PREDOKSI SENTIMEN:")
print("=" * 70)
for i, (sent, expected) in enumerate(test_cases, 1):
    res = predictor.predict(sent)
    pred_label = res['sentiment']
    conf = res['confidence']
    status = "PASSED" if pred_label == expected else "FAILED"
    print(f"Test #{i}:")
    print(f"  📝 Input    : {res['text_original']}")
    print(f"  🎯 Expected : {expected}")
    print(f"  🏷️ Predicted: {pred_label} (Confidence: {conf:.2%})")
    print(f"  ✅ Status   : {status}")
    print("-" * 70)